# NextAI — Qwen3.5-0.8B engineering student training

This notebook trains the single compact NextAI student. It downloads bounded CAD/BIM training samples, performs BF16/FP16 LoRA SFT, evaluates the held-out set, and exports a Q4_K_M GGUF. Teacher weights and raw training data are never copied into the Next3D installer.

**Runtime:** choose a GPU runtime. A T4-class GPU should be enough for the 0.8B LoRA run according to the Unsloth recipe, but check the available GPU and memory before proceeding. The notebook stops if CUDA is unavailable.

The first run is a controlled 800-step experiment, not a claim of engineering-grade performance. Native NextCAD/Next BIM validation is still required before promotion.

In [ ]:
import os, subprocess, sys, json, torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
REPO = '/content/Next3D'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','--branch','nextai-operator-foundation','https://github.com/28royalpapi/Next3D.git',REPO],check=True)
os.chdir(REPO)
print('Training branch:', subprocess.check_output(['git','branch','--show-current'],text=True).strip())

In [ ]:
%pip install --upgrade --quiet unsloth unsloth_zoo transformers trl datasets peft accelerate bitsandbytes pillow

## 1. Prepare the teaching material

The Autodesk Zero-to-CAD 100K set supplies multi-view CAD examples and construction-operation traces. The IFC/BIM instruction and QA sets supply BIM/IFC concepts. Next3D-authored lessons fill 2D drafting, parametric sketches, documentation and construction workflow gaps. Only bounded samples are downloaded.

In [ ]:
DATA_DIR = '/content/nextai-corpus'
subprocess.run([
    sys.executable, 'training/nextai/prepare_public_corpora.py',
    '--output-dir', DATA_DIR,
    '--max-cad-train', '2500', '--max-cad-validation', '250', '--max-cad-test', '250',
    '--max-bim-train', '30000', '--max-bim-validation', '3000',
    '--max-ifcqa-train', '5000', '--max-ifcqa-test', '1000',
    '--max-mixed-train', '24000', '--max-validation', '2500', '--max-test', '1200',
],check=True)
with open(DATA_DIR + '/manifest.json') as f: manifest=json.load(f)
print(json.dumps({k:manifest[k] for k in ['student','train_records','validation_records','test_records','training_domain_counts']},indent=2))

## 2. Train the compact student

The base is the post-trained Qwen3.5-0.8B checkpoint so the student retains its general instruction-following behaviour. We add a small LoRA adapter to specialise it. We do not use QLoRA for this Qwen3.5 recipe. The vision encoder remains frozen in this first pass; the model still receives CAD views while learning language-side CAD/BIM concepts and structured responses.

In [ ]:
OUTPUT_DIR = '/content/nextai-run-001'
subprocess.run([
    sys.executable, 'training/nextai/train_nextai.py',
    '--data-dir', DATA_DIR,
    '--output-dir', OUTPUT_DIR,
    '--base-model', 'Qwen/Qwen3.5-0.8B',
    '--max-steps', '800',
    '--max-seq-length', '4096',
    '--batch-size', '1',
    '--gradient-accumulation', '8',
    '--lora-rank', '16',
    '--save-merged-model',
],check=True)

## 2B. Compact the vision projector

Qwen3.5 is multimodal. The language-model GGUF is not the whole model: a separate vision projector is required for image input. This step converts the projector to Q8_0, verifies the GGUF header and checksum, and removes the larger full-precision projector only after the replacement passes validation.

In [ ]:
LLAMA_CPP = '/content/llama.cpp'
if not os.path.isdir(LLAMA_CPP):
    subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',LLAMA_CPP],check=True)
subprocess.run([
    sys.executable, 'training/nextai/compact_multimodal_export.py',
    '--merged-model-dir', OUTPUT_DIR + '/merged_16bit',
    '--llama-cpp-dir', LLAMA_CPP,
    '--gguf-dir', OUTPUT_DIR + '/gguf_q4_k_m',
],check=True)
with open(OUTPUT_DIR + '/gguf_q4_k_m/compact-export-manifest.json') as f: compact=json.load(f)
print('Compact model bytes:', compact['total_gguf_bytes'])
print(json.dumps(compact['files'],indent=2))

## 3. Compare the trained student against the untouched model

These are language/structure diagnostics only. The final acceptance test must execute proposed actions in the actual FreeCAD and BIMCraft engines and validate the resulting native projects.

In [ ]:
subprocess.run([
    sys.executable, 'training/nextai/evaluate_nextai.py',
    '--model', 'Qwen/Qwen3.5-0.8B', '--test-jsonl', DATA_DIR + '/test.jsonl',
    '--output', OUTPUT_DIR + '/evaluation-base.json', '--max-records', '200'
],check=True)
subprocess.run([
    sys.executable, 'training/nextai/evaluate_nextai.py',
    '--model', OUTPUT_DIR + '/adapter', '--test-jsonl', DATA_DIR + '/test.jsonl',
    '--output', OUTPUT_DIR + '/evaluation-tuned.json', '--max-records', '200'
],check=True)
for name in ['evaluation-base.json','evaluation-tuned.json']:
    with open(OUTPUT_DIR + '/' + name) as f: report=json.load(f)
    print('\n',name, json.dumps(report['metrics_by_domain'],indent=2))

## 4. Check the compact export size

The exported model is only a candidate. Do not replace the packaged model unless it passes the held-out benchmark and native CAD/BIM acceptance suite. This notebook reports the exact model-file sizes so the full Next3D release budget can be measured separately.

In [ ]:
result_path = OUTPUT_DIR + '/training_result.json'
with open(result_path) as f: result=json.load(f)
print('GGUF total bytes:', result.get('gguf_total_bytes', 'not exported'))
for item in result.get('gguf_files', []): print(item)
print('Candidate output:', OUTPUT_DIR)